In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver # InMemorySaver save the state value in memory


In [3]:
load_dotenv()

llm1 = ChatOllama(model= "nemotron-3-nano:30b-cloud")
llm2= ChatGroq(model="llama-3.3-70b-versatile")
llm3 = ChatOllama(
    model="smollm2:360m"
)

final_model = llm1.with_fallbacks(fallbacks=[llm2,llm3])

In [4]:
class JokeState(TypedDict):
    
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):
    prompt = f"Generate a joke on the topic {state["topic"]}"
    response = final_model.invoke(prompt).content

    return{
        "joke":response
    }

In [6]:
def generate_explain(state: JokeState):
    prompt = f"Write an explanation for the joke - {state["joke"]}"
    response = final_model.invoke(prompt).content

    return{
        "explanation":response
    }

In [7]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explain", generate_explain)


graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explain")
graph.add_edge("generate_explain", END)

checkpointer = InMemorySaver() # Checkpointer object bana liye hai 

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {
    "configurable":{"thread_id":"1"}
}
workflow.invoke(
    {
        "topic":"pizza"
    },
    config=config1
)

{'topic': 'pizza',
 'joke': "You know it's a great day to go to the pizza place, because the guy who gets the cheese and the tomato is so delicious. He's like a pizza, but even better!",
 'explanation': "You're not quite understanding the joke yet, but don't worry, it's quite simple. So, imagine a pizza place with a guy who gets a pizza made with cheese and tomato. Let's say that guy's name is Fred. Now, imagine someone who makes Fred even better by adding extra toppings to the pizza. That would be awesome, right? So, the joke is saying that a great day to go to a pizza place would be to grab a great pizza with a delicious Fred, or even better, a pizza with an extra-tasty Fred."}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "You know it's a great day to go to the pizza place, because the guy who gets the cheese and the tomato is so delicious. He's like a pizza, but even better!", 'explanation': "You're not quite understanding the joke yet, but don't worry, it's quite simple. So, imagine a pizza place with a guy who gets a pizza made with cheese and tomato. Let's say that guy's name is Fred. Now, imagine someone who makes Fred even better by adding extra toppings to the pizza. That would be awesome, right? So, the joke is saying that a great day to go to a pizza place would be to grab a great pizza with a delicious Fred, or even better, a pizza with an extra-tasty Fred."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ff2b-6800-6b2d-8002-2b05f1ad89b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-24T19:33:41.794282+00:00', parent_config={'configurable': {'thread_id': '1', 'checkp

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "You know it's a great day to go to the pizza place, because the guy who gets the cheese and the tomato is so delicious. He's like a pizza, but even better!", 'explanation': "You're not quite understanding the joke yet, but don't worry, it's quite simple. So, imagine a pizza place with a guy who gets a pizza made with cheese and tomato. Let's say that guy's name is Fred. Now, imagine someone who makes Fred even better by adding extra toppings to the pizza. That would be awesome, right? So, the joke is saying that a great day to go to a pizza place would be to grab a great pizza with a delicious Fred, or even better, a pizza with an extra-tasty Fred."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ff2b-6800-6b2d-8002-2b05f1ad89b7'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-24T19:33:41.794282+00:00', parent_config={'configurable': {'thread_id': '1', 'check

## Time Travel


In [13]:
workflow.get_state(
    {
        "configurable":{
            "thread_id":"1",
            'checkpoint_id': '1f19ff2a-c9fa-66e0-8000-69bf918295ec'
        }
    }

)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f19ff2a-c9fa-66e0-8000-69bf918295ec'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-24T19:33:25.224214+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19ff2a-c9f5-68ae-bfff-bae15b75bb76'}}, tasks=(PregelTask(id='23821e87-c7df-cdfc-5a9d-d8e5ca138f8f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "You know it's a great day to go to the pizza place, because the guy who gets the cheese and the tomato is so delicious. He's like a pizza, but even better!"}),), interrupts=())

In [ ]:
workflow.invoke(
    None,
    {
        "configurable":{
            "thread_id":"1",
            'checkpoint_id': '1f19ff2a-c9fa-66e0-8000-69bf918295ec'
        }
    }
)

## Update State

In [2]:
workflow.update_state(
    {
        "configurable":{
            "thread_id":"1",
            'checkpoint_id': '1f19ff2a-c9fa-66e0-8000-69bf918295ec'
        }
    },
    {
        "topic":"samosa"
    }
)

NameError: name 'workflow' is not defined